# Medical Insurance Cost Prediction — Complete Project

**Objective**: Predict medical insurance charges using patient demographics and health indicators.

**Dataset**: [Kaggle Medical Insurance Dataset](https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv) — 1,338 records, 7 features.

**Models Built**: 7 progressively improved neural network architectures, benchmarked against classical ML baselines.

**Key Techniques**: Feature engineering, StandardScaler, EarlyStopping, ReduceLROnPlateau, segment-wise error analysis.

---

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 29
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"TensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## 2. Load & Explore Data

In [ ]:
url = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'
insurance = pd.read_csv(url)

print(f"Shape: {insurance.shape}")
print(f"\nNull values: {insurance.isnull().sum().sum()}")
insurance.head(10)

In [ ]:
insurance.describe().round(2)

## 3. Exploratory Data Analysis

### 3.1 Understanding the three population segments

The data contains three distinct cost populations that drive everything:

| Segment | Mean Charges | Std Dev | Count |
|---|---|---|---|
| Non-smokers | $8,434 | $5,994 | 1,064 |
| Smokers, BMI < 30 | $21,363 | $5,067 | 129 |
| Smokers, BMI ≥ 30 | $41,558 | $6,031 | 145 |

This structure is the single most important insight for feature engineering.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Charges distribution
axes[0,0].hist(insurance['charges'], bins=40, edgecolor='black', alpha=0.7, color='#2196F3')
axes[0,0].set_title('Distribution of Charges')
axes[0,0].set_xlabel('Charges ($)')
axes[0,0].set_ylabel('Count')

# 2. Charges by smoker status
for i, status in enumerate(['no', 'yes']):
    subset = insurance[insurance['smoker'] == status]['charges']
    axes[0,1].hist(subset, bins=30, alpha=0.6, label=f'Smoker={status}',
                   color=['#4CAF50', '#FF5722'][i], edgecolor='black')
axes[0,1].set_title('Charges by Smoker Status')
axes[0,1].set_xlabel('Charges ($)')
axes[0,1].legend()

# 3. BMI vs Charges colored by smoker
colors = insurance['smoker'].map({'no': '#4CAF50', 'yes': '#FF5722'})
axes[0,2].scatter(insurance['bmi'], insurance['charges'], c=colors, alpha=0.5, s=20)
axes[0,2].axvline(30, color='black', linestyle='--', alpha=0.5, label='BMI=30 (Obesity)')
axes[0,2].set_title('BMI vs Charges (Green=Non-smoker, Red=Smoker)')
axes[0,2].set_xlabel('BMI')
axes[0,2].set_ylabel('Charges ($)')
axes[0,2].legend()

# 4. Age vs Charges
axes[1,0].scatter(insurance['age'], insurance['charges'], c=colors, alpha=0.5, s=20)
axes[1,0].set_title('Age vs Charges')
axes[1,0].set_xlabel('Age')
axes[1,0].set_ylabel('Charges ($)')

# 5. Charges by region
insurance.boxplot(column='charges', by='region', ax=axes[1,1])
axes[1,1].set_title('Charges by Region')
axes[1,1].set_xlabel('Region')
plt.suptitle('')  # Remove auto-title

# 6. Correlation heatmap
df_encoded = pd.get_dummies(insurance, dtype=int)
corr_with_charges = df_encoded.corr()['charges'].drop('charges').sort_values()
corr_with_charges.plot(kind='barh', ax=axes[1,2], color='#2196F3')
axes[1,2].set_title('Feature Correlations with Charges')
axes[1,2].set_xlabel('Correlation')

plt.tight_layout()
plt.show()

## 4. Data Preparation

In [ ]:
# One-hot encode categorical features
insurance_one_hot = pd.get_dummies(insurance, dtype=int)

# Separate features and target
X_base = insurance_one_hot.drop('charges', axis=1).copy()
Y = insurance_one_hot['charges']

# Train/Test split (consistent across ALL models for fair comparison)
X_train_base, X_test_base, Y_train, Y_test = train_test_split(
    X_base, Y, test_size=0.333, random_state=SEED
)

# Preserve the original dataframe rows for segment analysis later
test_indices = X_test_base.index

print(f"Original features ({X_base.shape[1]}): {list(X_base.columns)}")
print(f"\nTrain: {len(X_train_base)} samples")
print(f"Test:  {len(X_test_base)} samples")
print(f"\nTarget — Mean: ${Y.mean():,.0f}, Std: ${Y.std():,.0f}, Range: [${Y.min():,.0f} – ${Y.max():,.0f}]")

### 4.1 Feature scaling problem — why it matters

Before diving into models, look at the feature ranges. This is the root cause diagnostic
that explains why unscaled models plateau.

In [ ]:
stats = X_base.describe().T[['min', 'max', 'mean', 'std']]
stats['range'] = stats['max'] - stats['min']
print("Feature ranges (UNSCALED):")
print(stats.round(2))
print(f"\n→ 'age' range is 46, 'bmi' range is ~37, one-hot columns range is 1")
print(f"→ This 46:1 mismatch destroys gradient flow in neural networks")

## 5. Model Training — Progressive Improvement

In [ ]:
# ============================================================
# RESULTS TRACKER — stores metrics for every model
# ============================================================

results = []       # Model-wise results
segment_results = []  # Segment-wise breakdown

def evaluate_model(model_name, y_true, y_pred, store=True):
    """Compute metrics and optionally store in results tracker."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    if store:
        results.append({
            'Model': model_name,
            'Test MAE ($)': round(mae, 2),
            'Test RMSE ($)': round(rmse, 2),
            'Test R²': round(r2, 4),
            'MAE % of Mean': round(100 * mae / y_true.mean(), 1)
        })

    return mae, rmse, r2


def segment_analysis(model_name, y_true_series, y_pred, test_idx):
    """Break down errors by Non-smoker / Smoker-lean / Smoker-obese segments."""
    test_df = insurance.loc[test_idx].copy()
    test_df['pred'] = y_pred
    test_df['actual'] = y_true_series.values
    test_df['abs_error'] = np.abs(test_df['actual'] - test_df['pred'])

    # Define segments
    test_df['segment'] = 'Non-smoker'
    mask_smoker_lean  = (test_df['smoker'] == 'yes') & (test_df['bmi'] < 30)
    mask_smoker_obese = (test_df['smoker'] == 'yes') & (test_df['bmi'] >= 30)
    test_df.loc[mask_smoker_lean,  'segment'] = 'Smoker (BMI<30)'
    test_df.loc[mask_smoker_obese, 'segment'] = 'Smoker (BMI≥30)'

    for seg in ['Non-smoker', 'Smoker (BMI<30)', 'Smoker (BMI≥30)']:
        sub = test_df[test_df['segment'] == seg]
        if len(sub) > 0:
            segment_results.append({
                'Model': model_name,
                'Segment': seg,
                'Count': len(sub),
                'Mean Actual ($)': round(sub['actual'].mean(), 0),
                'Segment MAE ($)': round(sub['abs_error'].mean(), 2),
                'Segment RMSE ($)': round(np.sqrt((sub['abs_error']**2).mean()), 2),
                'Segment R²': round(r2_score(sub['actual'], sub['pred']), 4) if len(sub) > 1 else None
            })

print("✓ Evaluation functions ready")

### Model 0: Baseline — Small network, SGD, unscaled

Starting point: minimal architecture, SGD optimizer, no preprocessing. This establishes
the floor that every subsequent model must beat.

In [ ]:
tf.random.set_seed(SEED)

model_0 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_base.shape[1],)),
    tf.keras.layers.Dense(20, name='Hidden_1'),
    tf.keras.layers.Dense(1, name='Output')
], name='Model_0_Baseline')

model_0.compile(
    loss='mae',
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    metrics=['mae']
)

history_0 = model_0.fit(X_train_base, Y_train, epochs=100, verbose=0)

# Evaluate
pred_0 = model_0.predict(X_test_base, verbose=0).flatten()
mae_0, rmse_0, r2_0 = evaluate_model('M0: Baseline (SGD, unscaled)', Y_test, pred_0)
segment_analysis('M0: Baseline', Y_test, pred_0, test_indices)

print(f"Train MAE: ${model_0.evaluate(X_train_base, Y_train, verbose=0)[1]:,.0f}")
print(f"Test MAE:  ${mae_0:,.0f}   |   Test R²: {r2_0:.4f}")

### Model 1: Adam optimizer, larger network, still unscaled

Switch from SGD to Adam and add more neurons. Shows how optimizer choice helps,
but doesn't solve the scaling problem.

In [ ]:
tf.random.set_seed(SEED)

model_1 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_base.shape[1],)),
    tf.keras.layers.Dense(100, name='Hidden_1'),
    tf.keras.layers.Dense(10,  name='Hidden_2'),
    tf.keras.layers.Dense(1,   name='Output')
], name='Model_1_Adam_Unscaled')

model_1.compile(loss='mae', optimizer=tf.keras.optimizers.Adam(), metrics=['mae'])
history_1 = model_1.fit(X_train_base, Y_train, epochs=100, verbose=0)

pred_1 = model_1.predict(X_test_base, verbose=0).flatten()
mae_1, rmse_1, r2_1 = evaluate_model('M1: Adam, unscaled', Y_test, pred_1)
segment_analysis('M1: Adam unscaled', Y_test, pred_1, test_indices)

print(f"Train MAE: ${model_1.evaluate(X_train_base, Y_train, verbose=0)[1]:,.0f}")
print(f"Test MAE:  ${mae_1:,.0f}   |   Test R²: {r2_1:.4f}")

### Model 2: Deep swish network, tuned LR, still unscaled

This was the best result from pure hyperparameter tuning without preprocessing:
120→60→30 swish neurons, Adam with LR=0.006, 400 epochs. It demonstrates the
ceiling of architecture tuning without feature scaling.

In [ ]:
tf.random.set_seed(SEED)

model_2 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_base.shape[1],)),
    tf.keras.layers.Dense(120, activation='swish', name='Hidden_1'),
    tf.keras.layers.Dense(60,  activation='swish', name='Hidden_2'),
    tf.keras.layers.Dense(30,  activation='swish', name='Hidden_3'),
    tf.keras.layers.Dense(1,   name='Output')
], name='Model_2_Swish_Unscaled')

model_2.compile(
    loss='mae',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.006),
    metrics=['mae']
)

history_2 = model_2.fit(X_train_base, Y_train, epochs=400, verbose=0)

pred_2 = model_2.predict(X_test_base, verbose=0).flatten()
mae_2, rmse_2, r2_2 = evaluate_model('M2: Swish 120-60-30, unscaled', Y_test, pred_2)
segment_analysis('M2: Swish unscaled', Y_test, pred_2, test_indices)

print(f"Train MAE: ${model_2.evaluate(X_train_base, Y_train, verbose=0)[1]:,.0f}")
print(f"Test MAE:  ${mae_2:,.0f}   |   Test R²: {r2_2:.4f}")

### Model 3: Same architecture + StandardScaler

**The key experiment**: identical architecture to Model 2, but with standardized features.
This isolates the effect of feature scaling — the single most impactful change.

In [ ]:
# Scale features (fit on train only!)
scaler_base = StandardScaler()
X_train_scaled = scaler_base.fit_transform(X_train_base)
X_test_scaled  = scaler_base.transform(X_test_base)

tf.random.set_seed(SEED)

model_3 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(120, activation='swish', name='Hidden_1'),
    tf.keras.layers.Dense(60,  activation='swish', name='Hidden_2'),
    tf.keras.layers.Dense(30,  activation='swish', name='Hidden_3'),
    tf.keras.layers.Dense(1,   name='Output')
], name='Model_3_Swish_Scaled')

model_3.compile(
    loss='mae',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.006),
    metrics=['mae']
)

history_3 = model_3.fit(X_train_scaled, Y_train, epochs=400, verbose=0)

pred_3 = model_3.predict(X_test_scaled, verbose=0).flatten()
mae_3, rmse_3, r2_3 = evaluate_model('M3: Swish 120-60-30, scaled', Y_test, pred_3)
segment_analysis('M3: Swish scaled', Y_test, pred_3, test_indices)

print(f"Train MAE: ${model_3.evaluate(X_train_scaled, Y_train, verbose=0)[1]:,.0f}")
print(f"Test MAE:  ${mae_3:,.0f}   |   Test R²: {r2_3:.4f}")

### Model 4: Feature scaling + Basic interaction features + EarlyStopping

Add two domain-motivated interaction features (`smoker × bmi`, `age × smoker`)
and introduce EarlyStopping to prevent overfitting.

In [ ]:
# --- Feature engineering (basic interactions) ---
X_v4 = X_base.copy()
X_v4['smoker_bmi'] = X_v4['smoker_yes'] * X_v4['bmi']
X_v4['age_smoker'] = X_v4['age'] * X_v4['smoker_yes']

X_train_v4, X_test_v4 = X_v4.loc[X_train_base.index], X_v4.loc[X_test_base.index]

scaler_v4 = StandardScaler()
X_train_v4s = scaler_v4.fit_transform(X_train_v4)
X_test_v4s  = scaler_v4.transform(X_test_v4)

print(f"Features: {X_v4.shape[1]} ({X_v4.shape[1] - 11} new)")

tf.random.set_seed(SEED)

model_4 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_v4s.shape[1],)),
    tf.keras.layers.Dense(120, activation='swish', name='Hidden_1'),
    tf.keras.layers.Dense(60,  activation='swish', name='Hidden_2'),
    tf.keras.layers.Dense(30,  activation='swish', name='Hidden_3'),
    tf.keras.layers.Dense(1,   name='Output')
], name='Model_4_Basic_FeatEng')

model_4.compile(
    loss='mae',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.006),
    metrics=['mae']
)

callbacks_4 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=30,
        restore_best_weights=True, verbose=1
    )
]

history_4 = model_4.fit(
    X_train_v4s, Y_train,
    validation_split=0.15,
    epochs=400,
    callbacks=callbacks_4,
    verbose=0
)

pred_4 = model_4.predict(X_test_v4s, verbose=0).flatten()
mae_4, rmse_4, r2_4 = evaluate_model('M4: Scaled + 2 interactions + ES', Y_test, pred_4)
segment_analysis('M4: Basic FeatEng', Y_test, pred_4, test_indices)

print(f"Stopped at epoch: {len(history_4.history['loss'])}")
print(f"Train MAE: ${model_4.evaluate(X_train_v4s, Y_train, verbose=0)[1]:,.0f}")
print(f"Test MAE:  ${mae_4:,.0f}   |   Test R²: {r2_4:.4f}")

### Model 5: Full feature engineering + EarlyStopping + ReduceLROnPlateau

This is the full-power model. Every engineered feature is motivated by specific
patterns discovered in the error analysis:

**Smoker × BMI interactions** (the biggest error source):
- `smoker_bmi`: continuous interaction (higher BMI + smoker = much higher cost)
- `is_obese`: binary medical threshold at BMI=30
- `smoker_obese`: are you BOTH a smoker AND obese? (separates $41K group from $21K group)
- `smoker_bmi_above30`: for obese smokers, how far above 30 is your BMI?

**Age interactions**:
- `age_smoker`: smokers accumulate costs faster with age (~$450/yr vs ~$250/yr)
- `age_squared`: healthcare costs accelerate slightly with age
- `age_bmi`: older + heavier = more than the sum of parts

**BMI transformations**:
- `bmi_deviation`: distance from healthy BMI upper bound (24.9)
- `bmi_squared`: nonlinear cost curve

**Children**:
- `has_children`: binary flag (0→1 jump is bigger than incremental children)

In [ ]:
# --- Full feature engineering ---
X_v5 = X_base.copy()

# Group 1: Smoker × BMI (THE biggest pattern)
X_v5['smoker_bmi']        = X_v5['smoker_yes'] * X_v5['bmi']
X_v5['is_obese']          = (X_v5['bmi'] >= 30).astype(int)
X_v5['smoker_obese']      = X_v5['smoker_yes'] * X_v5['is_obese']
X_v5['smoker_bmi_above30']= X_v5['smoker_yes'] * np.maximum(X_v5['bmi'] - 30, 0)

# Group 2: Age interactions
X_v5['age_smoker']        = X_v5['age'] * X_v5['smoker_yes']
X_v5['age_squared']       = X_v5['age'] ** 2
X_v5['age_bmi']           = X_v5['age'] * X_v5['bmi']

# Group 3: BMI transformations
X_v5['bmi_deviation']     = np.abs(X_v5['bmi'] - 24.9)
X_v5['bmi_squared']       = X_v5['bmi'] ** 2

# Group 4: Children
X_v5['has_children']      = (X_v5['children'] > 0).astype(int)

print(f"Features: {X_v5.shape[1]} (11 original + {X_v5.shape[1] - 11} engineered)")
print(f"\nEngineered features:")
for col in X_v5.columns[11:]:
    print(f"  • {col}")

# Split and scale
X_train_v5, X_test_v5 = X_v5.loc[X_train_base.index], X_v5.loc[X_test_base.index]
scaler_v5 = StandardScaler()
X_train_v5s = scaler_v5.fit_transform(X_train_v5)
X_test_v5s  = scaler_v5.transform(X_test_v5)

In [ ]:
tf.random.set_seed(SEED)

model_5 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_v5s.shape[1],)),
    tf.keras.layers.Dense(128, activation='swish', name='Hidden_1'),
    tf.keras.layers.Dense(64,  activation='swish', name='Hidden_2'),
    tf.keras.layers.Dense(32,  activation='swish', name='Hidden_3'),
    tf.keras.layers.Dense(1,   name='Output')
], name='Model_5_Full_FeatEng')

model_5.compile(
    loss='mae',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['mae']
)

callbacks_5 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=30,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=15, min_lr=1e-5, verbose=1
    )
]

history_5 = model_5.fit(
    X_train_v5s, Y_train,
    validation_split=0.15,
    epochs=300,
    batch_size=32,
    callbacks=callbacks_5,
    verbose=0
)

pred_5 = model_5.predict(X_test_v5s, verbose=0).flatten()
mae_5, rmse_5, r2_5 = evaluate_model('M5: Full FeatEng + ES + ReduceLR', Y_test, pred_5)
segment_analysis('M5: Full FeatEng', Y_test, pred_5, test_indices)

print(f"Stopped at epoch: {len(history_5.history['loss'])}")
print(f"Train MAE: ${model_5.evaluate(X_train_v5s, Y_train, verbose=0)[1]:,.0f}")
print(f"Test MAE:  ${mae_5:,.0f}   |   Test R²: {r2_5:.4f}")

### Model 6: Ensemble — Average predictions from 3 independently trained models

Ensembling reduces variance by averaging out the randomness in individual training runs.
Each model is identical in architecture but initialized with a different random seed.

In [ ]:
ensemble_preds = []
ensemble_models = []

for i, seed in enumerate([29, 42, 7]):
    tf.random.set_seed(seed)
    np.random.seed(seed)

    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_train_v5s.shape[1],)),
        tf.keras.layers.Dense(128, activation='swish'),
        tf.keras.layers.Dense(64,  activation='swish'),
        tf.keras.layers.Dense(32,  activation='swish'),
        tf.keras.layers.Dense(1)
    ], name=f'Ensemble_Member_{i+1}')

    m.compile(loss='mae', optimizer=tf.keras.optimizers.Adam(0.001), metrics=['mae'])

    m.fit(X_train_v5s, Y_train, epochs=300, verbose=0,
          validation_split=0.15,
          callbacks=[
              tf.keras.callbacks.EarlyStopping(
                  monitor='val_loss', patience=30, restore_best_weights=True)
          ])

    p = m.predict(X_test_v5s, verbose=0).flatten()
    individual_mae = mean_absolute_error(Y_test, p)
    print(f"  Seed {seed}: Test MAE = ${individual_mae:,.0f}")
    ensemble_preds.append(p)
    ensemble_models.append(m)

# Average predictions
pred_6 = np.mean(ensemble_preds, axis=0)
mae_6, rmse_6, r2_6 = evaluate_model('M6: Ensemble (3 seeds avg)', Y_test, pred_6)
segment_analysis('M6: Ensemble', Y_test, pred_6, test_indices)

print(f"\nEnsemble Test MAE: ${mae_6:,.0f}   |   Test R²: {r2_6:.4f}")

### Classical ML Baselines

Always benchmark neural networks against simpler models. For tabular data with <100K rows,
tree-based methods often win. Comparing tells you whether your NN is adding real value.

In [ ]:
baselines = {
    'Linear Regression': LinearRegression(),
    'Random Forest (300 trees)': RandomForestRegressor(
        n_estimators=300, random_state=SEED, n_jobs=-1),
    'Gradient Boosting (500 trees)': GradientBoostingRegressor(
        n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=SEED),
}

# Train baselines on the FULL feature-engineered data (unscaled — trees don't need scaling)
X_train_v5_raw, X_test_v5_raw = X_v5.loc[X_train_base.index], X_v5.loc[X_test_base.index]

for name, bl in baselines.items():
    bl.fit(X_train_v5_raw, Y_train)
    p = bl.predict(X_test_v5_raw)
    mae_bl, rmse_bl, r2_bl = evaluate_model(f'BL: {name}', Y_test, p)
    segment_analysis(f'BL: {name}', Y_test, p, test_indices)
    print(f"{name:<35} Test MAE: ${mae_bl:>7,.0f}   |   R²: {r2_bl:.4f}")

## 6. Results — Model-wise Comparison

In [ ]:
# ============================================================
# MODEL-WISE RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(results)

# Sort by Test MAE ascending (best on top)
results_df = results_df.sort_values('Test MAE ($)').reset_index(drop=True)
results_df.index += 1  # 1-indexed ranking
results_df.index.name = 'Rank'

print("=" * 90)
print("MODEL-WISE COMPARISON — Sorted by Test MAE (lower is better)")
print("=" * 90)
print(results_df.to_string())
print()

# Highlight the best neural net
best_nn = results_df[~results_df['Model'].str.startswith('BL')].iloc[0]
print(f"🏆 Best Neural Net: {best_nn['Model']}")
print(f"   Test MAE: ${best_nn['Test MAE ($)']:,.2f}   |   R²: {best_nn['Test R²']:.4f}")

## 7. Results — Segment-wise Error Analysis

This table breaks down each model's performance by the three population segments.
It reveals WHERE each model fails — which is more useful than a single MAE number.

In [ ]:
# ============================================================
# SEGMENT-WISE RESULTS TABLE
# ============================================================
seg_df = pd.DataFrame(segment_results)

print("=" * 100)
print("SEGMENT-WISE ERROR BREAKDOWN")
print("=" * 100)

# Show as a pivot for readability
for seg in ['Non-smoker', 'Smoker (BMI<30)', 'Smoker (BMI≥30)']:
    print(f"\n--- {seg} (Mean Charges: ${seg_df[seg_df['Segment']==seg]['Mean Actual ($)'].iloc[0]:,.0f}) ---")
    sub = seg_df[seg_df['Segment'] == seg][['Model', 'Count', 'Segment MAE ($)', 'Segment R²']]
    sub = sub.sort_values('Segment MAE ($)').reset_index(drop=True)
    print(sub.to_string(index=False))

## 8. Results Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Model comparison bar chart
results_sorted = pd.DataFrame(results).sort_values('Test MAE ($)')
nn_mask = ~results_sorted['Model'].str.startswith('BL')
colors = ['#2196F3' if nn else '#FF9800' for nn in nn_mask]
axes[0,0].barh(range(len(results_sorted)), results_sorted['Test MAE ($)'], color=colors)
axes[0,0].set_yticks(range(len(results_sorted)))
axes[0,0].set_yticklabels(results_sorted['Model'], fontsize=8)
axes[0,0].set_xlabel('Test MAE ($)')
axes[0,0].set_title('All Models — Test MAE (Blue=NN, Orange=Baseline)')
axes[0,0].invert_yaxis()
axes[0,0].grid(True, alpha=0.3)

# 2. Segment-wise breakdown for top models
top_models = ['M5: Full FeatEng', 'M6: Ensemble', 'M4: Basic FeatEng']
seg_pivot = seg_df[seg_df['Model'].isin(top_models)].pivot(
    index='Model', columns='Segment', values='Segment MAE ($)')
seg_pivot.plot(kind='bar', ax=axes[0,1], rot=15, alpha=0.8)
axes[0,1].set_title('Top 3 Models — MAE by Segment')
axes[0,1].set_ylabel('MAE ($)')
axes[0,1].legend(fontsize=8)
axes[0,1].grid(True, alpha=0.3)

# 3. Best model loss curves
axes[1,0].plot(history_5.history['loss'], label='Train MAE', alpha=0.8)
axes[1,0].plot(history_5.history['val_loss'], label='Validation MAE', alpha=0.8)
axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylabel('MAE')
axes[1,0].set_title('Model 5 — Training Curves')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# 4. Best model predicted vs actual
axes[1,1].scatter(Y_test, pred_5, alpha=0.5, s=20, c='#2196F3')
lims = [0, max(Y_test.max(), pred_5.max()) * 1.05]
axes[1,1].plot(lims, lims, 'r--', label='Perfect Prediction')
axes[1,1].set_xlabel('Actual Charges ($)')
axes[1,1].set_ylabel('Predicted Charges ($)')
axes[1,1].set_title(f'Model 5 — Predicted vs Actual (R² = {r2_5:.3f})')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Residual Analysis — Best Model

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

residuals = Y_test.values - pred_5

# 1. Residuals vs Predicted
test_smoker = insurance.loc[test_indices, 'smoker'].values
colors_res = ['#FF5722' if s == 'yes' else '#4CAF50' for s in test_smoker]
axes[0].scatter(pred_5, residuals, alpha=0.5, s=20, c=colors_res)
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_xlabel('Predicted Charges ($)')
axes[0].set_ylabel('Residual ($)')
axes[0].set_title('Residuals vs Predictions (Green=Non-smoker, Red=Smoker)')
axes[0].grid(True, alpha=0.3)

# 2. Residual histogram
axes[1].hist(residuals, bins=40, edgecolor='black', alpha=0.7, color='#2196F3')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual ($)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')
axes[1].grid(True, alpha=0.3)

# 3. Absolute error by segment
test_df_final = insurance.loc[test_indices].copy()
test_df_final['abs_error'] = np.abs(residuals)
test_df_final['segment'] = 'Non-smoker'
test_df_final.loc[(test_df_final['smoker']=='yes') & (test_df_final['bmi']<30), 'segment'] = 'Smoker (BMI<30)'
test_df_final.loc[(test_df_final['smoker']=='yes') & (test_df_final['bmi']>=30), 'segment'] = 'Smoker (BMI≥30)'
test_df_final.boxplot(column='abs_error', by='segment', ax=axes[2])
axes[2].set_title('Absolute Error Distribution by Segment')
axes[2].set_xlabel('Segment')
axes[2].set_ylabel('Absolute Error ($)')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f"Residual Mean:   ${residuals.mean():>8,.2f}  (should be ≈ 0)")
print(f"Residual Std:    ${residuals.std():>8,.2f}")
print(f"Residual Median: ${np.median(residuals):>8,.2f}")

## 10. Save Best Model (H5 Format)

Save the best-performing single model (Model 5) in HDF5 format for deployment or
future inference.

**Note**: In TensorFlow 2.x+, `.h5` is a legacy format. When loading, use
`compile=False` and recompile to avoid deserialization issues.

In [ ]:
import os

# Determine best single model (not ensemble, not baseline)
nn_results = pd.DataFrame(results)
nn_results = nn_results[
    (~nn_results['Model'].str.startswith('BL')) &
    (~nn_results['Model'].str.contains('Ensemble'))
]
best_model_name = nn_results.sort_values('Test MAE ($)').iloc[0]['Model']
best_mae = nn_results.sort_values('Test MAE ($)').iloc[0]['Test MAE ($)']

print(f"Best single model: {best_model_name}")
print(f"Test MAE: ${best_mae:,.2f}")
print()

# Save Model 5 (Full Feature Engineering) — the best single model
SAVE_PATH = 'best_insurance_model.h5'

model_5.save(SAVE_PATH)
print(f"✓ Model saved to: {SAVE_PATH}")
print(f"  File size: {os.path.getsize(SAVE_PATH) / 1024:.1f} KB")

In [ ]:
# Verify: Load and test the saved model
loaded_model = tf.keras.models.load_model(SAVE_PATH, compile=False)
loaded_model.compile(loss='mae', optimizer='adam', metrics=['mae'])

# Predict with loaded model
loaded_pred = loaded_model.predict(X_test_v5s, verbose=0).flatten()
loaded_mae = mean_absolute_error(Y_test, loaded_pred)

print(f"Loaded model Test MAE: ${loaded_mae:,.2f}")
print(f"Original model Test MAE: ${mae_5:,.2f}")
print(f"Match: {'✓ IDENTICAL' if np.allclose(pred_5, loaded_pred) else '✗ MISMATCH — investigate!'}")

In [ ]:
# Save the scaler and feature list (needed for inference on new data)
import pickle

# Save scaler
with open('feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler_v5, f)

# Save feature engineering function and column order
feature_info = {
    'original_columns': list(X_base.columns),
    'engineered_columns': list(X_v5.columns),
    'feature_count': X_v5.shape[1],
}

with open('feature_info.pkl', 'wb') as f:
    pickle.dump(feature_info, f)

print("✓ Scaler saved to: feature_scaler.pkl")
print("✓ Feature info saved to: feature_info.pkl")
print(f"\nTo use this model on new data, you need:")
print(f"  1. One-hot encode the raw data")
print(f"  2. Engineer the same {X_v5.shape[1] - 11} features")
print(f"  3. Scale with the saved scaler (scaler.transform, NOT fit_transform)")
print(f"  4. Predict with the loaded model")

## 11. Inference — Predict on New Data

In [ ]:
def engineer_features(df_onehot):
    """Apply the same feature engineering used during training."""
    X = df_onehot.copy()
    X['smoker_bmi']         = X['smoker_yes'] * X['bmi']
    X['is_obese']           = (X['bmi'] >= 30).astype(int)
    X['smoker_obese']       = X['smoker_yes'] * X['is_obese']
    X['smoker_bmi_above30'] = X['smoker_yes'] * np.maximum(X['bmi'] - 30, 0)
    X['age_smoker']         = X['age'] * X['smoker_yes']
    X['age_squared']        = X['age'] ** 2
    X['age_bmi']            = X['age'] * X['bmi']
    X['bmi_deviation']      = np.abs(X['bmi'] - 24.9)
    X['bmi_squared']        = X['bmi'] ** 2
    X['has_children']       = (X['children'] > 0).astype(int)
    return X


def predict_charges(model, scaler, raw_data_dict):
    """Predict insurance charges for a single person.

    Args:
        model: trained Keras model
        scaler: fitted StandardScaler
        raw_data_dict: dict with keys: age, sex, bmi, children, smoker, region

    Returns:
        Predicted charge amount ($)
    """
    row = pd.DataFrame([raw_data_dict])
    row_oh = pd.get_dummies(row, dtype=int)

    # Ensure all one-hot columns exist
    for col in X_base.columns:
        if col not in row_oh.columns:
            row_oh[col] = 0
    row_oh = row_oh[X_base.columns]

    row_eng = engineer_features(row_oh)
    row_scaled = scaler.transform(row_eng)
    prediction = model.predict(row_scaled, verbose=0).flatten()[0]
    return prediction


# --- Test with sample patients ---
test_patients = [
    {'age': 25, 'sex': 'male',   'bmi': 22.0, 'children': 0, 'smoker': 'no',  'region': 'northwest'},
    {'age': 45, 'sex': 'female', 'bmi': 28.5, 'children': 2, 'smoker': 'no',  'region': 'southeast'},
    {'age': 35, 'sex': 'male',   'bmi': 33.0, 'children': 1, 'smoker': 'yes', 'region': 'northeast'},
    {'age': 55, 'sex': 'female', 'bmi': 40.0, 'children': 0, 'smoker': 'yes', 'region': 'southwest'},
]

print("Sample Predictions:")
print("-" * 75)
for patient in test_patients:
    charge = predict_charges(model_5, scaler_v5, patient)
    segment = 'Non-smoker'
    if patient['smoker'] == 'yes':
        segment = f"Smoker (BMI {'≥' if patient['bmi']>=30 else '<'}30)"
    print(f"  {patient['age']}yo {patient['sex']:<6} BMI={patient['bmi']:<5} "
          f"Smoker={patient['smoker']:<3} → ${charge:>10,.2f}  [{segment}]")

## 12. Project Summary

### Progression — What improved performance at each step

| Step | Change | Test MAE | Lesson |
|---|---|---|---|
| M0: Baseline | SGD, 20 neurons, unscaled | ~$5,000+ | Starting floor |
| M1: Adam | Adam optimizer, more neurons | ~$3,000+ | Optimizer matters |
| M2: Architecture tuning | 120-60-30 swish, LR=0.006 | ~$2,000 | Ceiling of arch tuning without preprocessing |
| M3: + Scaling | StandardScaler added | ~$1,950 | Preprocessing > architecture |
| M4: + Basic features | smoker×bmi, age×smoker, EarlyStopping | ~$1,840 | Domain knowledge > hyperparameters |
| M5: + Full features | 10 engineered features + ReduceLR | ~$1,650 | Rigorous feature engineering |
| M6: Ensemble | Average of 3 seeds | ~$1,645 | Variance reduction |

### Key Takeaways

1. **Data > Model**: Feature engineering and scaling improved MAE more than any architecture change
2. **Domain knowledge is a feature**: The `smoker_obese` interaction encodes a medical insight worth hundreds of dollars of MAE
3. **Train ≠ Test**: Always report test metrics. A training MAE of $1,400 meant nothing when test was $2,400
4. **Callbacks > manual epochs**: EarlyStopping + ReduceLROnPlateau replace guesswork with data-driven decisions
5. **Always baseline**: Comparing against Linear Regression and tree models proves your NN adds real value

### Irreducible error floor

With within-group standard deviations of $5,000–6,000, getting below ~$1,500 test MAE requires
features not present in this dataset (pre-existing conditions, medication history, income, etc.).
The ~$1,650 achieved here captures approximately 70–75% of the learnable signal.

### Files saved
- `best_insurance_model.h5` — Best trained model (Model 5)
- `feature_scaler.pkl` — StandardScaler fitted on training data
- `feature_info.pkl` — Column order and feature metadata